# Arithmetic GPT — starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s8-arith-gpt.ipynb)

Teach a fixed tiny GPT (2 layers, 0.42 M parameters) to multiply two 4-digit numbers:
[challenge 196](https://ml-arena.com/viewchallenge/196) on ML-Arena. You submit **only the weights**;
what you design is **the text the model is trained to write** after the prompt `1234*5678=`.

This notebook runs end-to-end: **download → choose a training format → train → score on the dev set → submit**.
It trains the *answer-only* format, which scores 0%. Your job starts in section 5.

## 0. Setup

Use a GPU runtime: Runtime → Change runtime type → T4 GPU.

In [ ]:
!pip install -q "mlarena-sdk>=3" safetensors

import mlarena, csv, math, random, sys, time, torch

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page)
CHALLENGE_ID = 196

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Get the files

`arith_gpt.py` is the model, the greedy decoder, the answer parser and the scorer: the leaderboard
imports the same file. `dev_problems.csv` holds 200 products with their answers.

In [ ]:
client.download_dataset(CHALLENGE_ID, "data/")
sys.path.insert(0, "data")
import arith_gpt as ag

with open("data/dev_problems.csv") as fh:
    dev = [(int(r["a"]), int(r["b"]), int(r["answer"])) for r in csv.DictReader(fh)]
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(len(dev), "dev problems, e.g.", dev[0], "- device", device)
print("vocabulary:", ag.VOCAB)

## 2. The training format

The model reads the prompt `a*b=` and writes a **completion**. The grader takes the number after the
**last `#`**, and the completion must end with a newline (`\n`) within 128 tokens. Everything before the `#`
is yours: a scratchpad, or nothing.

`completion(a, b)` is the only thing this notebook asks you to change. It starts as *answer only*.

In [ ]:
def completion(a, b):
    return f"#{a * b}\n"

for a, b, _ in dev[:3]:
    print(repr(f"{a}*{b}=" + completion(a, b)))

A training example is prompt + completion. The loss is computed on the completion only (label `-100`
on the prompt, as in *Instruction Tuning and PEFT*): the model learns to answer, not to write prompts.

In [ ]:
def make_batch(rng, n, completion):
    seqs, plen = [], []
    for _ in range(n):
        a, b = rng.randint(1000, 9999), rng.randint(1000, 9999)
        prompt = f"{a}*{b}="
        seqs.append(ag.encode(prompt + completion(a, b)))
        plen.append(len(prompt))
    T = max(map(len, seqs)) - 1
    x = torch.full((n, T), ag.PAD)
    y = torch.full((n, T), -100)
    for i, s in enumerate(seqs):
        s = torch.tensor(s)
        x[i, :len(s) - 1] = s[:-1]
        y[i, plen[i] - 1:len(s) - 1] = s[plen[i]:]   # targets: the completion only
    return x.to(device), y.to(device)

## 3. Train

From scratch, for a fixed number of **minutes**, so two formats are compared at the same budget.
Every `eval_every` seconds it prints the loss and the two leaderboard metrics on the dev set:
**exact** (the ranking) and **per digit** (the share of answer digits that are right).

In [ ]:
def train(completion, minutes=5, lr=1e-3, batch=256, eval_every=60, seed=0):
    torch.manual_seed(seed); rng = random.Random(seed)
    model = ag.GPT().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1, betas=(0.9, 0.98))
    budget, t0, step, next_eval = minutes * 60, time.time(), 0, eval_every
    while (elapsed := time.time() - t0) < budget:
        frac = elapsed / budget   # warmup, then cosine decay over the budget
        for g in opt.param_groups:
            g["lr"] = lr * min(1, (step + 1) / 100) * (0.1 + 0.45 * (1 + math.cos(math.pi * frac)))
        x, y = make_batch(rng, batch, completion)
        loss = torch.nn.functional.cross_entropy(model(x).flatten(0, 1), y.flatten())
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); step += 1
        if elapsed > next_eval:
            r = ag.evaluate_problems(model, dev)
            print(f"{elapsed / 60:4.1f} min  step {step:5d}  loss {loss.item():.4f}  "
                  f"exact {r['accuracy']:.1%}  per digit {r['digit_accuracy']:.1%}")
            next_eval += eval_every
    return model.eval()

model = train(completion, minutes=5)

## 4. Score it and look at what it writes

In [ ]:
r = ag.evaluate_problems(model, dev)
print(f"dev: exact {r['accuracy']:.1%}   per digit {r['digit_accuracy']:.1%}")

x = torch.tensor([ag.encode(f"{a}*{b}=") for a, b, _ in dev[:5]], device=device)
for (a, b, c), row in zip(dev[:5], model.generate(x).tolist()):
    text = ag.decode(row).split("\n")[0]
    print(f"{a}*{b}={text:<60} parsed {ag.parse_answer(row)}   true {c}")

**Questions**
- Which digits of the answer does the answer-only model get right? Why those?
- Is it the model that is too small, or the task that asks too much of each token?

## 5. Your turn: design the completion

Rewrite `completion(a, b)`, retrain, compare the dev scores at the same number of minutes. Check first that your format is correct: `ag.parse_answer` must return the product, within 128 tokens.

In [ ]:
def completion(a, b):
    return f"#{a * b}\n"    # <-- your scratchpad goes before the '#'

for a, b, c in dev[:200]:
    s = completion(a, b)
    assert len(s) <= ag.MAX_NEW_TOKENS and ag.parse_answer(ag.encode(s)) == c, s
print(repr(completion(1234, 5678)), len(completion(1234, 5678)), "tokens")

In [ ]:
model = train(completion, minutes=10)

**Hints**, one at a time (the challenge page explains each):
1. Tokens written before the `#` are extra computation: the model reads its own previous tokens.
2. Write what you would write on paper: one partial product per digit of `b`.
3. Keep each step local: a running sum after each partial product.
4. The model writes left to right, but carries travel from right to left.
5. Exact accuracy can stay at 0% while the loss falls, then jump: watch the loss.

## 6. Submit

The grader loads only the weights: save them with `ag.save_weights`.

In [ ]:
ag.save_weights(model.cpu(), "weights.safetensors")
result = client.submit(challenge_id=CHALLENGE_ID, files=["weights.safetensors"], submission_name="answer-only")
print(result)
model.to(device);